In [1]:
# ── JSON → CSV Extractor (registry-based) ─────────────────────────────────────
# Reads your existing run registry CSV, matches each row to its JSON file
# by timestamp, and produces three output CSVs.
# ─────────────────────────────────────────────────────────────────────────────
import json, glob, os
import pandas as pd
import numpy as np

REGISTRY_CSV = (
    "/home/rayudu/My_work1/IPD/work/forge/llm/IPD-LLM-Agents3/results/"
    "table-3b1b37f4-5697-4464-90ea-7337893fc6e5.csv"
)
JSON_FOLDER  = "/home/rayudu/My_work1/IPD/work/forge/llm/IPD-LLM-Agents3/results"
OUTPUT_DIR   = "/home/rayudu/My_work1/IPD/work/forge/llm/IPD-LLM-Agents3/csv_output1"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Model mix label (short) ───────────────────────────────────────────────────
def mix_label(m0, m1, m2):
    short = lambda m: ("L" if "llama" in m.lower()
                       else "G" if "gemma" in m.lower()
                       else "M" if "mistral" in m.lower()
                       else "X")
    from collections import Counter
    letters = [short(m0), short(m1), short(m2)]
    counts  = Counter(letters)
    return "+".join(f"{v}{k}" for k, v in sorted(counts.items(), key=lambda x: -x[1]))

# ── Step 1: load registry CSV ─────────────────────────────────────────────────
registry = pd.read_csv(REGISTRY_CSV, encoding="utf-8-sig")
registry.columns = registry.columns.str.strip()
print(f"Registry rows: {len(registry)}")
print(f"Columns: {list(registry.columns)}")

# ── Step 2: build timestamp → filepath map ────────────────────────────────────
ts_map = {}   # "2026-03-30T21:47:36" → "/path/to/file.json"
for fp in sorted(glob.glob(f"{JSON_FOLDER}/*.json")):
    try:
        with open(fp, encoding="utf-8") as f:
            d = json.load(f)
        ts = d.get("timestamp", "")[:19]   # first 19 chars: "2026-03-30T21:47:36"
        if ts:
            ts_map[ts] = fp
    except Exception as e:
        print(f"  WARN: could not read {fp}: {e}")

print(f"\nIndexed {len(ts_map)} JSON files in root folder")

# ── Step 3: open CSV writers ──────────────────────────────────────────────────
registry_rows = []    # enriched registry (file-level)
episode_rows  = []    # episode-level
round_rows    = []    # round-level (no text — see below)
round_text_rows = []  # round-level WITH reasoning text

# ── Step 4+5+6+7: process each registry row ───────────────────────────────────
matched = 0; unmatched = 0

for _, reg_row in registry.iterrows():

    # Convert CSV timestamp to JSON format for lookup
    csv_ts  = str(reg_row["Timestamp"]).strip()            # "2026-03-30 21:47:36"
    json_ts = csv_ts.replace(" ", "T")                      # "2026-03-30T21:47:36"

    fp = ts_map.get(json_ts)
    if fp is None:
        print(f"  UNMATCHED: {csv_ts}  comment={reg_row.get('Comment','?')}")
        unmatched += 1
        continue

    try:
        with open(fp, encoding="utf-8") as f:
            d = json.load(f)
    except Exception as e:
        print(f"  ERROR reading {fp}: {e}")
        continue

    matched += 1
    cfg      = d.get("config", {})
    fname    = os.path.basename(fp)
    comment  = d.get("comment", reg_row.get("Comment",""))
    m0       = cfg.get("model_0", "")
    m1       = cfg.get("model_1", "")
    m2       = cfg.get("model_2", "")
    mix      = mix_label(m0, m1, m2)
    T        = cfg.get("temperature")
    HW       = cfg.get("history_window_size")
    reset    = cfg.get("reset_between_episodes")
    retries  = cfg.get("force_decision_retries")

    # ── Shared identification block ───────────────────────────────────────────
    ident = {
        "json_filename":            fname,
        "comment":                  comment,
        "model_mix_label":          mix,
        "temperature":              T,
        "history_window":           HW,
        "reset_between_episodes":   reset,
        "decision_retries":         retries,
        "model_0_full":             m0,
        "model_1_full":             m1,
        "model_2_full":             m2,
    }

    # ── File-level agent outcomes ─────────────────────────────────────────────
    file_row = dict(ident)
    file_row.update({
        "elapsed_seconds":          d.get("elapsed_seconds", ""),
        "hostname":                 d.get("hostname", ""),
        "num_episodes":             cfg.get("num_episodes"),
        "rounds_per_episode":       cfg.get("rounds_per_episode"),
        "decision_token_limit":     cfg.get("decision_token_limit"),
        "reflection_token_limit":   cfg.get("reflection_token_limit"),
        "reflection_type":          cfg.get("reflection_type"),
    })
    ag_rates = []
    for i in range(3):
        ag = d.get(f"agent_{i}", {})
        file_row[f"agent_{i}_total_score"]        = ag.get("total_score", "")
        file_row[f"agent_{i}_total_cooperations"] = ag.get("total_cooperations", "")
        file_row[f"agent_{i}_overall_coop_rate"]  = ag.get("overall_cooperation_rate", "")
        if "overall_cooperation_rate" in ag:
            ag_rates.append(ag["overall_cooperation_rate"])
    file_row["mean_group_coop_rate"] = float(np.mean(ag_rates)) if ag_rates else ""
    registry_rows.append(file_row)

    # ── Episode-level ─────────────────────────────────────────────────────────
    for ep in d.get("episodes", []):
        ep_row = dict(ident)
        ep_row["episode_id"] = ep.get("episode", "")
        ep_rates = []
        for i in range(3):
            ag = ep.get(f"agent_{i}", {})
            ep_row[f"agent_{i}_episode_score"]    = ag.get("episode_score", "")
            ep_row[f"agent_{i}_cooperations"]     = ag.get("cooperations", "")
            ep_row[f"agent_{i}_cooperation_rate"] = ag.get("cooperation_rate", "")
            ep_row[f"agent_{i}_reflection"]       = (
                ag.get("reflection", "").replace("\n", " ").strip()
            )
            if "cooperation_rate" in ag:
                ep_rates.append(ag["cooperation_rate"])
        if ep_rates:
            ep_row["mean_group_coop_rate"] = float(np.mean(ep_rates))
            ep_row["coop_asymmetry"]       = float(np.std(ep_rates))
        else:
            ep_row["mean_group_coop_rate"] = ""
            ep_row["coop_asymmetry"]       = ""
        episode_rows.append(ep_row)

        # ── Round-level ───────────────────────────────────────────────────────
        for rnd in ep.get("rounds", []):
            rnd_base = dict(ident)
            rnd_base["episode_id"] = ep.get("episode", "")
            rnd_base["round_id"]   = rnd.get("round", "")
            for i in range(3):
                rnd_base[f"agent_{i}_action"]  = rnd.get(f"agent_{i}_action", "")
                rnd_base[f"agent_{i}_payoff"]  = rnd.get(f"agent_{i}_payoff", "")
            # Numeric-only version (no text)
            round_rows.append(dict(rnd_base))
            # Text version — add reasoning
            rnd_text = dict(rnd_base)
            for i in range(3):
                rnd_text[f"agent_{i}_reasoning"] = (
                    rnd.get(f"agent_{i}_reasoning", "").replace("\n", " ").strip()
                )
            round_text_rows.append(rnd_text)

print(f"\nMatched: {matched}  |  Unmatched: {unmatched}")

# ── Step 8: write CSVs ────────────────────────────────────────────────────────
df_reg   = pd.DataFrame(registry_rows)
df_ep    = pd.DataFrame(episode_rows)
df_round = pd.DataFrame(round_rows)
df_rtext = pd.DataFrame(round_text_rows)

reg_path   = os.path.join(OUTPUT_DIR, "enriched_registry.csv")
ep_path    = os.path.join(OUTPUT_DIR, "episode_level.csv")
round_path = os.path.join(OUTPUT_DIR, "round_level_no_text.csv")
rtext_path = os.path.join(OUTPUT_DIR, "round_level_with_text.csv")

df_reg.to_csv(reg_path,   index=False, encoding="utf-8")
df_ep.to_csv(ep_path,     index=False, encoding="utf-8")
df_round.to_csv(round_path, index=False, encoding="utf-8")
df_rtext.to_csv(rtext_path, index=False, encoding="utf-8")

print(f"\n── Output Summary ──")
print(f"  enriched_registry.csv      : {len(df_reg):>6} rows  {len(df_reg.columns)} cols  → {reg_path}")
print(f"  episode_level.csv          : {len(df_ep):>6} rows  {len(df_ep.columns)} cols  → {ep_path}")
print(f"  round_level_no_text.csv    : {len(df_round):>6} rows  {len(df_round.columns)} cols  → {round_path}")
print(f"  round_level_with_text.csv  : {len(df_rtext):>6} rows  {len(df_rtext.columns)} cols  → {rtext_path}")

Registry rows: 79
Columns: ['Timestamp', 'Comment', 'Temperature', 'History Window', 'Reset Between Episodes', 'Models (Player 0, 1, 2)', 'Decision Retries']

Indexed 91 JSON files in root folder

Matched: 79  |  Unmatched: 0

── Output Summary ──
  enriched_registry.csv      :     79 rows  27 cols  → /home/rayudu/My_work1/IPD/work/forge/llm/IPD-LLM-Agents3/csv_output1/enriched_registry.csv
  episode_level.csv          :   3950 rows  25 cols  → /home/rayudu/My_work1/IPD/work/forge/llm/IPD-LLM-Agents3/csv_output1/episode_level.csv
  round_level_no_text.csv    :  79000 rows  18 cols  → /home/rayudu/My_work1/IPD/work/forge/llm/IPD-LLM-Agents3/csv_output1/round_level_no_text.csv
  round_level_with_text.csv  :  79000 rows  21 cols  → /home/rayudu/My_work1/IPD/work/forge/llm/IPD-LLM-Agents3/csv_output1/round_level_with_text.csv
